In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob
from scipy.optimize import curve_fit

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
df = pd.read_csv("../../data/processed/fig/fig2_geo_reg_moransi_results.csv")
df

In [ ]:
# --- Assume data is prepared ---
df_long = df.melt(id_vars='ratio', 
                  value_vars=['Ours', 'Non-spatial'], 
                  var_name='Model', 
                  value_name='R2')

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))

palette = {'Ours': '#8E8BFE', 'Non-spatial': '#5FB7B9'}
hue_order = ['Ours', 'Non-spatial']

# 1. Boxplot (bottom layer).
sns.boxplot(
    data=df_long, 
    x='ratio', 
    y='R2', 
    hue='Model', 
    hue_order=hue_order,
    palette=palette,
    ax=ax,
    width=0.6,          # narrower boxes
    linewidth=1.2,
    showfliers=False,   # hide outliers
    boxprops=dict(alpha=1) 
)

# 2. Median connector line (top layer).
sns.pointplot(
    data=df_long, 
    x='ratio', 
    y='R2', 
    hue='Model', 
    hue_order=hue_order,
    palette=palette,
    ax=ax,
    estimator=np.median,
    dodge=0.3,       # set to 0.4 to center on the boxes
    markers=['o', 'D'], 
    linestyles=['-', '--'],
    errorbar=None,
    scale=0.8,
    legend=False
)

# --- Polish ---
ax.set_ylabel('Overall $R^2$', fontsize=18)
ax.set_xlabel('Average surveyed areas (%)', fontsize=18)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.grid(axis='both', color='gray', linestyle='--', alpha=0.2)
ax.set_axisbelow(True) # grid below data

ax.set_ylim(0, 1)

# Legend.
handles, labels = ax.get_legend_handles_labels()
# Take the first two handles (from the boxplot).
ax.legend(handles=handles[:2], labels=labels[:2],
          loc='lower right', fontsize=16, frameon=False)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig2_geo_reg_compare.svg')
plt.show()

In [ ]:
df50 = df[df['ratio'] == 50].copy()
df50

In [ ]:
import scipy.stats as stats
from matplotlib.lines import Line2D

In [ ]:
# --- Assume df50 is the input ---
# Drop NaNs.
plot_data = df50.dropna(subset=['Moran_I', 'Ours', 'Non-spatial'])

# 1. Pearson correlation.
r_ours, p_ours = stats.pearsonr(plot_data['Moran_I'], plot_data['Ours'])
r_base, p_base = stats.pearsonr(plot_data['Moran_I'], plot_data['Non-spatial'])

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 7))

sns.regplot(
    data=plot_data,
    x='Moran_I',
    y='Ours',
    scatter=True,
    color=palette['Ours'],
    scatter_kws={'alpha': 0.6, 's': 60, 'edgecolor': 'white', 'linewidths': 0.5},
    line_kws={'color': palette['Ours'], 'linewidth': 3},
    ax=ax
)

sns.regplot(
    data=plot_data,
    x='Moran_I',
    y='Non-spatial',
    scatter=True,
    color=palette['Non-spatial'],
    scatter_kws={'alpha': 0.6, 's': 60, 'edgecolor': 'white', 'linewidths': 0.5},
    line_kws={'color': palette['Non-spatial'], 'linewidth': 3},
    ax=ax
)

# 4. Polish axes.
ax.set_ylabel('Overall $R^2$', fontsize=16, fontweight='bold', labelpad=10)
ax.set_xlabel("Spatial autocorrelation (Moran's I)", fontsize=18, fontweight='bold', labelpad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.tick_params(axis='both', which='major', labelsize=18, width=2)

# 5. Add custom legend.
legend_elements = [
    Line2D([0], [0], color=palette['Ours'], lw=3, label=f'Ours ($r={r_ours:.2f}, p<0.001$)'),
    Line2D([0], [0], color=palette['Non-spatial'], lw=3, label=f'Non-spatial ($r={r_base:.2f}, p<0.001$)')
]

ax.legend(handles=legend_elements, loc='upper left', frameon=False, fontsize=18)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig2_morani.svg')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.regplot(
    data=df50,
    x='Moran_I',
    y='delta',
    scatter=True,
    ax=ax,
    line_kws={'color': 'red'}
)
plt.show()